# Mean based pixelwise fusion for base predictions.

In [1]:
import os
from pathlib import Path

root_is_cwd = os.getcwd().endswith("fusionLearning")

if not root_is_cwd:
    os.chdir(Path().resolve().parent)
    print("Changed to root directory")
    root_is_cwd = True
else:
    print("Already in root directory")

print(os.getcwd())

Changed to root directory
C:\Users\GAMER01\codeproj\fusionLearning\fusionLearning


In [2]:
from data.fusion_dataloader import FusionDataset
from fusion.means import PixelwiseMeanFusion

from torch.utils.data import DataLoader, random_split
from segmentation_models_pytorch import Unet, UnetPlusPlus, FPN, Linknet, Segformer

from config import CUB_SEGMENTATIONS
from models.consts import BATCHSIZE

c:\Users\GAMER01\codeproj\fusionLearning\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
arch_encoder_pairings = { 
    Unet : ["resnet34", "resnet18"],
    UnetPlusPlus : ["resnet34", "resnet18"],
    Linknet : ["resnet18"],
    FPN : ["resnet34"],
    Segformer : ["resnet18"],
}
def archToString(arch):
    if arch is Unet:
        return 'Unet'
    elif arch is UnetPlusPlus:
        return 'UnetPlusPlus'
    elif arch is Linknet:
        return 'Linknet'
    elif arch is FPN:
        return 'FPN'
    elif arch is Segformer:
        return 'Segformer'
    else:
        raise ValueError(f"Unknown architecture: {arch}")

MODEL_NAMES = [archToString(arch) + "_" + encoder for arch, encoders in arch_encoder_pairings.items() for encoder in encoders]



## Create the dataloaders

In [4]:
import torch

fdataset = FusionDataset(model_names=MODEL_NAMES, segmentation_dir=CUB_SEGMENTATIONS)

generator : torch.Generator = torch.Generator().manual_seed(42)


numsamples = len(fdataset)
trainsize = int(0.7 * numsamples)
valsize = int(0.2 * numsamples)
testsize = numsamples - trainsize - valsize

ftrain_dataset, fval_dataset, ftest_dataset = random_split(fdataset, [trainsize, valsize, testsize], generator=generator)

ftrain_loader = DataLoader(ftrain_dataset, batch_size=BATCHSIZE, shuffle=True)
fval_loader = DataLoader(fval_dataset, batch_size=BATCHSIZE, shuffle=False)
ftest_loader = DataLoader(ftest_dataset, batch_size=BATCHSIZE, shuffle=False)


## Debugging

In [5]:
verbose = False

if verbose:
    for p in fdataset.segmentation_filenames:
        print(p)

In [6]:
debug_model_prediction_sizes = False

if debug_model_prediction_sizes:
    # write contents to .txt
    for i, p in enumerate(fdataset.model_segmentations_filenames):
        for j in p:
            with open(f'model {i}.txt', 'a') as f:
                f.write(j + '\n')
                
    # read sizes and output to stdio
    for i in range(len(fdataset.model_segmentations_paths)):
        with open(f'model {i}.txt', 'r') as f:
            chars = len(f.read())
        print(f'Amount of characters in model {i}.txt: {chars}')


In [7]:
debug_shape = False

if debug_shape:
    for i, (preds, mask, filename) in enumerate(fdataset):
        print(preds.shape)
        print(mask.shape)


### Quick visualization step before fusing results (compares baseline predictions to ground truth)

In [8]:
visualizing = False

In [9]:
import matplotlib.pyplot as plt
from IPython.display import clear_output
import time

if visualizing:
    for i, (preds, mask, fn) in enumerate(ftest_loader):    
        # Remove the batch dimension since it's 1
        preds = preds.squeeze(0)  # Shape: [7, 3, H, W]
        mask = mask.squeeze(0)    # Shape: [1, H, W]
        
        for j in range(preds.shape[0]):  # Loop through the 7 frames
            try:
                # Get j-th prediction and transpose to [H, W, 3] for matplotlib
                pred = preds[j].permute(1, 2, 0).cpu().numpy()
                
                plt.figure(figsize=(12, 6))
                
                # Show prediction
                plt.subplot(1, 2, 1)
                plt.imshow(pred)
                plt.title(f"Frame {j+1} - Prediction")
                
                # Show mask
                plt.subplot(1, 2, 2)
                plt.imshow(mask[0].cpu().numpy())
                plt.title("Ground Truth Mask")
                
                # Show filename if available
                if fn and len(fn) > 0:
                    print(f"Filename: {fn[0] if isinstance(fn, (list, tuple)) else fn}")
                
                plt.tight_layout()
                plt.show()
                # time.sleep(0.07)
                clear_output(wait=True)
                
            except Exception as e:
                print(f"Error processing frame {j}: {str(e)}")
                break



## Visualize fusion results for pixelwise approaches

### Arithmetic Mean Pixelwise

In [10]:
import matplotlib.pyplot as plt
import time
from IPython.display import clear_output

def fuse_and_visualize(fuser, loader):

    for i, (preds, mask, filename) in enumerate(loader):
        # Fuse the predictions (assuming preds has shape [batch_size, num_models, C, H, W])
        print("before", preds.shape)
        fused_mask = fuser.forward(preds)  # This should return [batch_size, C, H, W]
        print("after", fused_mask.shape)
        
        # Convert tensors to numpy for visualization
        fused_np = fused_mask[0].permute(1, 2, 0).cpu().numpy()  # Convert to [H, W, C] for matplotlib
        gt_mask_np = mask[0][0].cpu().numpy()  # Remove channel dimension for grayscale
        
        plt.figure(figsize=(12, 6))
        
        # Show fused prediction
        plt.subplot(1, 2, 1)
        plt.imshow(fused_np)
        plt.title(f"Fused Prediction - {fuser.name}")
        
        # Show ground truth mask
        plt.subplot(1, 2, 2)
        plt.imshow(gt_mask_np, cmap='gray')
        plt.title("Ground Truth Mask")
        
        # Show filename if available
        if filename:
            print(f"Filename: {filename[0] if isinstance(filename, (list, tuple)) else filename}")
        
        plt.tight_layout()
        plt.show()
        time.sleep(0.07)
        clear_output(wait=True)
    

### Measure IoU accuracy, compare to base models

In [11]:
import torch 

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)


cuda


In [12]:
from torchmetrics.classification import BinaryAUROC
import torch
from tqdm import tqdm

def measure_auroc(fuser, loader, device):
    # Initialize BinaryAUROC metric
    metric = BinaryAUROC(thresholds=32).to(device)
    
    with torch.no_grad():
        for i, (preds, mask, fn) in enumerate(tqdm(loader)):
            # Move data to device
            preds = preds.to(device)
            mask = mask.to(device).float()  # AUROC expects float for binary classification
            
            # Get fused prediction
            fused = fuser(preds)  # Shape: [1, 1, H, W]
            
            # Flatten predictions and mask for AUROC
            fused_flat = fused.view(-1)  # Flatten to 1D
            mask_flat = mask.view(-1)    # Flatten to 1D
            
            # Update metric - AUROC expects probabilities (no thresholding needed)
            metric.update(fused_flat, mask_flat)
    
    # Calculate final AUROC
    auroc_score = metric.compute()
    print(f"AUROC Score: {auroc_score:.4f}")
    
    # Reset the metric for future use
    metric.reset()
    return auroc_score

In [13]:
from fusion.means import MeanTypes, PixelwiseMeanFusion

fuser_am = PixelwiseMeanFusion("arithmeticMeanPixelwise", mean_type=MeanTypes.ARITHMETIC)
fuser_gm = PixelwiseMeanFusion("geometricMeanPixelwise", mean_type=MeanTypes.GEOMETRIC)
fuser_hm = PixelwiseMeanFusion("harmonicMeanPixelwise", mean_type=MeanTypes.HARMONIC)
fuser_pm = PixelwiseMeanFusion("powerMeanPixelwise", mean_type=MeanTypes.POWER, power=1.5)
fuser_mm = PixelwiseMeanFusion("medianMeanPixelwise", mean_type=MeanTypes.MEDIAN)
fuser_rms = PixelwiseMeanFusion("rootMeanSquarePixelwise", mean_type=MeanTypes.ROOT_MEAN_SQUARE)


In [14]:
fuse_and_visualize(fuser_gm, ftrain_loader)


before torch.Size([1, 7, 1, 480, 384])


KeyboardInterrupt: 

In [16]:
am_auroc = measure_auroc(fuser_am, fval_loader, device=device)
gm_auroc = measure_auroc(fuser_gm, fval_loader, device=device)
hm_auroc = measure_auroc(fuser_hm, fval_loader, device=device)
power_auroc = measure_auroc(fuser_pm, fval_loader, device=device)
median_auroc = measure_auroc(fuser_mm, fval_loader, device=device)
rms_auroc = measure_auroc(fuser_rms, fval_loader, device=device)

print(f"AM AUROC: {am_auroc}")
print(f"GM AUROC: {gm_auroc}")
print(f"HM AUROC: {hm_auroc}")
print(f"Power AUROC: {power_auroc}")
print(f"Median AUROC: {median_auroc}")
print(f"RMS AUROC: {rms_auroc}")



100%|██████████| 2357/2357 [00:34<00:00, 68.31it/s]


AUROC Score: 0.9556


100%|██████████| 2357/2357 [00:34<00:00, 67.97it/s]


AUROC Score: 0.9499


100%|██████████| 2357/2357 [00:34<00:00, 67.72it/s]


...

AUROC Score: 0.9555


100%|██████████| 2357/2357 [00:33<00:00, 69.42it/s]


AUROC Score: 0.9529


100%|██████████| 2357/2357 [00:34<00:00, 68.42it/s]

AUROC Score: 0.9552
AM AUROC: 0.9556106328964233
GM AUROC: 0.9499346613883972
HM AUROC: 0.940493106842041
Power AUROC: 0.955504834651947
Median AUROC: 0.9529253244400024
RMS AUROC: 0.955177903175354


In [ ]:
from fusion.interfaces import FusionModule
from config import FUSED_MODEL_SEGMENTATIONS

def generate_fusion_masks( fuser : FusionModule, 
                           dataloader : torch.utils.data.DataLoader,
                           save_dir : str) -> None:
    
    if os.path.exists(os.path.join(save_dir, fuser.name)):
        print(f"Segmentation masks for {fuser.name} already exist, skipping.")
        return

    os.makedirs(os.path.join(save_dir, fuser.name), exist_ok=True)

    for i, (predictions, mask, filename) in enumerate(dataloader):
        fused_mask : torch.Tensor = fuser.forward(predictions=predictions)

        img = Image.fromarray((fused_mask.cpu().numpy() * 255).astype(np.float32))
        img.save(os.path.join(save_dir, fuser.name, filename[0]))
        


In [ ]:
# generate masks here
from config import FUSED_MODEL_SEGMENTATIONS


# generate_fusion_masks(fuser_gm, ftrain_dataset, FUSED_MODEL_SEGMENTATIONS)

ValueError: unknown file extension: 